**Mount Drive in Colab and install dependencies**

In [ ]:
# 1. Mount Google Drive 
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repo
import os 
repo_path = '/content/darts-cards'
if not os.path.exists(repo_path):
    !git clone https://github.com/javiergarciaduran/darts-cards.git {repo_path}
else: 
    !git -C {repo_path} pull
%cd {repo_path}

In [ ]:
# 3. Install extra dependencies 
!pip install -q graphviz wandb
!apt-get install -q graphviz

d


In [ ]:
# 4. Symlink data from Drive into repo
!ln -sfn /content/drive/MyDrive/cards ./data/cards

# 5. Verify that the dataset is correctly found
!python -c "from datasets.cards import get_cards; tr, va, nc = get_cards('./data/cards'); print(f'train={len(tr)}; val = {len(va)}; classes = {nc}')"

**Perform the search over the different architectures (might take several hours to complete)**

In [ ]:
# 1. Run search
!python search.py \
    --name cards_search_v1 \
    --dataset cards \
    --data_path ./data/cards \
    --batch_size 64 \
    --init_channels 16 \
    --layers 8 \
    --epochs 50 \
    --w_lr 0.025 \
    --w_lr_min 0.001 \
    --alpha_lr 3e-4 \
    --alpha_weight_decay 1e-3 \
    --print_freq 50 \
    --seed 42 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/search_v1.log

Before running next cell:

1. Check the search log above for skip-connection collapse
2. Open experiments/search_cards_v1/normal.png and reduce.png
3. Copy the genotype from experiments/search_cards_v1/genotype.txt
4. Paste it into genotypes.py as CARDS_V1
5. Commit the change: git add genotypes.py && git commit -m "add CARDS_V1 genotype"
6. Then run Cell 7

In [ ]:
#2. Copy result genotype into genotypes.py
!python augment.py --name cards_augment_v1 --dataset cards \
    --data_path ./data/cards --batch_size 96 \
    --init_channels 24 --layers 14 --epochs 200 \
    --genotype CARDS_V1 \
    2>&1 | tee /content/drive/MyDrive/darts_logs/augment_v1.log

In [ ]:
# 3. Sync experiments to Drive 
!rsync -a --progress ./experiments/ /content/drive/MyDrive/darts_experiments/